# 🚗 차량 손상 YOLO 학습 — Google Colab (GPU)

로컬 M1(MPS)보다 5~10배 빠른 무료 GPU(T4)로 학습합니다.

## 준비 (최초 1회)
1. 로컬에서 데이터셋 압축: `cd ~/Desktop/car-damage-analyzer/data && zip -r yolo_dataset.zip yolo_dataset`
2. `yolo_dataset.zip` 을 구글드라이브 `MyDrive/` 에 업로드
3. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
4. 이 노트북을 위에서부터 순서대로 실행

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 라이브러리 설치 + 코드 클론

In [ ]:
!pip install ultralytics -q

# GitHub 코드 클론
GITHUB_REPO = "https://github.com/EunSeok-222/CarCheck.git"
!git clone $GITHUB_REPO
%cd CarCheck

## 3. 구글드라이브 마운트 + 데이터셋 압축해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 드라이브의 zip → 저장소 data/ 로 압축해제
!unzip -q /content/drive/MyDrive/yolo_dataset.zip -d data/
!ls data/yolo_dataset/images/train | wc -l
!ls data/yolo_dataset/images/val | wc -l

## 4. Colab 경로용 yaml 생성

In [ ]:
yaml_text = '''path: /content/car-damage-analyzer/data/yolo_dataset
train: images/train
val:   images/val
nc: 4
names:
  0: Scratched
  1: Breakage
  2: Separated
  3: Crushed
'''
with open('data/car_damage_colab.yaml', 'w') as f:
    f.write(yaml_text)
print('✅ yaml 생성 완료')

## 5. 학습 (GPU)

GPU라 batch를 16으로 키우고 epochs 100 으로 다시 제대로 학습합니다.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')
results = model.train(
    data='data/car_damage_colab.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # GPU
    patience=25,
    project='models',
    name='car_damage_seg',
    exist_ok=True,
)
print('🎉 학습 완료')
print('Best mAP50(Box): ', results.results_dict.get('metrics/mAP50(B)', 0))
print('Best mAP50(Mask):', results.results_dict.get('metrics/mAP50(M)', 0))

## 6. 결과 그래프 + best.pt 드라이브에 저장

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage('models/car_damage_seg/results.png'))

In [ ]:
import shutil
# best.pt 를 드라이브에 저장 (로컬로 다운받기 위해)
shutil.copy('models/car_damage_seg/weights/best.pt', '/content/drive/MyDrive/best.pt')
print('✅ best.pt → 드라이브 저장 완료')

# 또는 브라우저로 직접 다운로드
from google.colab import files
files.download('models/car_damage_seg/weights/best.pt')

## 학습 후 로컬 적용

드라이브/다운로드받은 `best.pt` 를 로컬 `car-damage-analyzer/models/best.pt` 에 덮어쓰면
Streamlit 앱이 자동으로 새 모델을 사용합니다.